# Week 3, Lab 4 — Crew + tools


In [1]:
import zipfile
import os

zip_path = "/content/shared.zip"      # Path of the uploaded ZIP file
extract_path = "/content/shared"      # Folder where files will be extracted

# Create the folder if it doesn't exist
os.makedirs(extract_path, exist_ok=True)

# Unzip
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ ZIP extracted successfully!")
print("Files extracted to:", extract_path)

✅ ZIP extracted successfully!
Files extracted to: /content/shared


In [2]:
import zipfile
import os


zip_path = "/content/shared.zip"
extract_path = "/content"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extracted successfully!")

✅ Extracted successfully!


In [3]:
WEEK = 'Week 3'
LAB = 'Lab 4 — crew tools'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


Week 3 / Lab 4 — crew tools
Environment: Google Colab
Backend: huggingface
Tip: Runtime → Change runtime type → T4 GPU for faster generation.
If import failed, unzip/clone the WHOLE course folder (not a single notebook).


In [4]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate fastapi uvicorn crewai
else:
    %pip install -q crewai ollama


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.9/198.9 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.4/269.4 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9

In [ ]:
from crewai import LLM

GEMINI_API_KEY = "your Gemini API key here"  # Replace with your actual Gemini API key

llm = LLM(
    model="gemini/gemini-3.6-flash",
    api_key=GEMINI_API_KEY
)

print("CrewAI is using Gemini 3.6 Flash")

CrewAI is using Gemini 3.6 Flash


In [7]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import BaseTool

# ------------------ Local Functions ------------------

def lookup_fact(topic: str) -> str:
    facts = {
        "langgraph": "LangGraph is a framework for building stateful, multi-agent AI applications using graphs. It helps agents maintain memory and manage complex workflows."
    }
    return facts.get(topic.lower(), "No fact found for this topic.")

def calculator(expression: str) -> str:
    try:
        return str(eval(expression))
    except Exception:
        return "Invalid expression."

# ------------------ Custom Tools ------------------

class LookupTool(BaseTool):
    name: str = "lookup_fact"
    description: str = "Look up a local fact about agentic AI topics."

    def _run(self, topic: str) -> str:
        return lookup_fact(topic)


class CalcTool(BaseTool):
    name: str = "calculator"
    description: str = "Evaluate arithmetic like '12*8+3'."

    def _run(self, expression: str) -> str:
        return calculator(expression)

# ------------------ Agents ------------------

researcher = Agent(
    role="Researcher",
    goal="Use lookup_fact for topic facts.",
    backstory="Librarian of the local knowledge base.",
    llm=llm,
    tools=[LookupTool()],
)

analyst = Agent(
    role="Analyst",
    goal="Use calculator for any math.",
    backstory="Likes numbers.",
    llm=llm,
    tools=[CalcTool()],
)

# ------------------ Tasks ------------------

t1 = Task(
    description="What is LangGraph? Use the lookup_fact tool.",
    expected_output="1-2 sentences grounded in the tool output.",
    agent=researcher,
)

t2 = Task(
    description="Compute 45*12+30 using the calculator tool and include the result in one closing sentence.",
    expected_output="A short recap including the calculated number.",
    agent=analyst,
)

# ------------------ Crew ------------------

crew = Crew(
    agents=[researcher, analyst],
    tasks=[t1, t2],
    process=Process.sequential,
)

# ✅ Google Colab / Jupyter
result = await crew.kickoff_async()
print(result)

LangGraph provides a robust graph-based framework for constructing stateful, multi-agent AI applications with integrated memory and workflow control. Evaluating the given expression 45 * 12 + 30 results in a total of 570.




**Next:** 3-agent research → draft → review.
